# Figure 2 — Tree Features ↔ Linear-Algebra Features (HBM verification)

Empirical verification of three Thesis V.04 identities on a sweep over $(\eta, m)$ with $\alpha \in \{1.0, 0.99, 0.95, 0.90\}$:

1. $\mu(U) = (1+\eta)/2$ — coherence of the rank-2 subspace $U=[v^{(0)}, v^{(1)}]$.
2. $\lambda_2 = m \cdot S_{\text{out}}^{\max}$ — Fiedler eigenvalue.
3. $\Delta\lambda \geq n_{\min}\rho - \eta\, n_{\min}\, S_{\text{out}}^{\max}$ — **Lemma 0.4**.

For the Hierarchical Block Model (HBM) the structural margin uses the *weakest* within-clan similarity, i.e. the deepest within-clan pair:
$$
S_{\text{in}}^{\min} \;=\; S_{\text{in}}\,\alpha^{D_{\max}}, \qquad D_{\max} \;=\; \lceil \log_2 \max(n_1, n_2) \rceil,
$$
so $\rho = S_{\text{in}}^{\min} - S_{\text{out}}^{\max}$. With this correction `gap_slack ≥ 0` holds across all swept $\alpha$.

Each $S$ is cached on disk under `cache/theoretical_interpretation/full/`, keyed by `hier_cbm_alpha…_eta…_n…_s_in…_s_out…` — re-running the notebook is free.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "setup.py").exists():
    ROOT = ROOT.parent
PROJECT_ROOT = ROOT / "sub_sampled_fielder_vec"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.theoretical_interpretation.utils import (
    build_decay_cbm_S, compute_lemma04_row,
    compute_fiedler_of_S, make_full_key, get_or_compute_full,
)

S_IN, S_OUT = 0.9, 0.05
ALPHA       = 0.99
ETA_VALUES  = [1, 2, 4, 8]
M_VALUES    = [90, 180, 360, 720, 1440, 2880, 5760]
ALPHAS_DEMO = [1.0, 0.99, 0.95, 0.90]
CACHE_DIR   = PROJECT_ROOT / "cache" / "theoretical_interpretation"


In [ ]:
def _clan_sizes(eta_target: int, m: int) -> tuple[int, int]:
    n1 = m // (1 + int(eta_target))
    return n1, m - n1

per_em: dict[tuple[int, int], tuple[np.ndarray, int, int]] = {}
for eta in ETA_VALUES:
    for m in M_VALUES:
        n1, n2 = _clan_sizes(eta, m)
        full_key = make_full_key(
            "hier_cbm", eta=eta, n=m, s_in=S_IN, s_out=S_OUT, alpha=ALPHA,
        )

        def builder(n1=n1, n2=n2, full_key=full_key, eta=eta, m=m):
            S = build_decay_cbm_S(n1, n2, S_IN, S_OUT, alpha=ALPHA)
            return S, compute_fiedler_of_S(S), {
                "tree_model": "hier_cbm", "eta": eta, "n1": n1, "n2": n2, "m": m,
                "s_in": S_IN, "s_out": S_OUT, "alpha": ALPHA, "full_key": full_key,
            }

        S, _ = get_or_compute_full(CACHE_DIR, full_key, builder)
        per_em[(eta, m)] = (S, n1, n2)
print(f"grid built: {len(per_em)} (eta, m) pairs, alpha={ALPHA}")


In [ ]:
df = pd.DataFrame([
    compute_lemma04_row(S, n1, n2, S_IN, S_OUT, ALPHA)
    for (S, n1, n2) in per_em.values()
])
assert (df["gap_slack"] >= -1e-9).all(), df[df["gap_slack"] < 0]
print(f"gap_slack range : [{df['gap_slack'].min():.4f}, {df['gap_slack'].max():.4f}]")
print(f"D_max range     : {sorted(df['D_max'].unique())}")
print(f"S_in_min range  : [{df['S_in_min'].min():.4f}, {df['S_in_min'].max():.4f}]")
df[["eta", "m", "n_min", "D_max", "S_in_min", "rho", "gap_emp", "gap_lb", "gap_slack"]].head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cmap = plt.get_cmap("viridis")
eta_to_color = {eta: cmap(i / max(1, len(ETA_VALUES) - 1)) for i, eta in enumerate(ETA_VALUES)}

def _identity_scatter(ax, x_col, y_col, *, log=False):
    for eta in ETA_VALUES:
        sub = df[df["eta"] == eta]
        ax.scatter(sub[x_col], sub[y_col], s=55, color=eta_to_color[eta],
                   edgecolors="k", linewidths=0.4, label=fr"$\eta={int(eta)}$", zorder=3)
    lo = float(min(df[x_col].min(), df[y_col].min()))
    hi = float(max(df[x_col].max(), df[y_col].max()))
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="y = x")
    if log:
        ax.set_xscale("log"); ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=9)

_identity_scatter(axes[0], "mu_pred", "mu_emp", log=True)
axes[0].set_xlabel(r"prediction  $(1+\eta)/2$")
axes[0].set_ylabel(r"empirical  $\mu(U)$")
axes[0].set_title(r"Coherence:  $\mu(U)_{\mathrm{emp}}$  vs  $(1+\eta)/2$")

_identity_scatter(axes[1], "lambda2_pred", "lambda2_emp", log=True)
axes[1].set_xlabel(r"prediction  $m \cdot S_{\mathrm{out}}^{\max}$")
axes[1].set_ylabel(r"empirical  $\lambda_2$")
axes[1].set_title(r"Fiedler eigenvalue:  $\lambda_{2,\mathrm{emp}}$  vs  $m\, S_{\mathrm{out}}^{\max}$")

_identity_scatter(axes[2], "gap_lb", "gap_emp", log=False)
axes[2].set_xlabel(r"lower bound  $n_{\min}(\rho - \eta\, S_{\mathrm{out}}^{\max})$")
axes[2].set_ylabel(r"empirical  $\Delta\lambda$")
axes[2].set_title(
    r"Spectral gap (Lemma 0.4):  $\Delta\lambda_{\mathrm{emp}}$  vs LB,  "
    r"$S_{\mathrm{in}}^{\min} = S_{\mathrm{in}}\alpha^{D_{\max}}$"
)

fig.suptitle(
    fr"HBM verification across $(\eta, m)$  —  "
    fr"$S_{{\mathrm{{in}}}}={S_IN}$, $S_{{\mathrm{{out}}}}={S_OUT}$, $\alpha={ALPHA}$  ({len(df)} points)",
    y=1.02,
)
fig.tight_layout()
plt.show()


In [ ]:
from matplotlib.ticker import MaxNLocator

DEMO_ETA, DEMO_M = 4, 360
n1_demo = DEMO_M // (1 + int(DEMO_ETA))
n2_demo = DEMO_M - n1_demo

# Heatmap S(α) and per-α scalar features at the demo (η, m).
demo_S = {a: build_decay_cbm_S(n1_demo, n2_demo, S_IN, S_OUT, alpha=a) for a in ALPHAS_DEMO}
df_demo = pd.DataFrame([
    {"alpha": a, **compute_lemma04_row(demo_S[a], n1_demo, n2_demo, S_IN, S_OUT, a)}
    for a in ALPHAS_DEMO
]).set_index("alpha")

# Slack curves: extended η grid (6 points) at fixed m.
# slack = Δλ_emp − (n_min·ρ − η·n_min·S_out^max). Stays ≥ 0 ⇔ Lemma 0.4 holds.
ETA_SLACK = [1, 2, 4, 8, 16, 32]
GAP_M = max(M_VALUES)
lemma_curves: dict[float, pd.DataFrame] = {}
for a in ALPHAS_DEMO:
    rows_a = []
    for eta in ETA_SLACK:
        n1 = GAP_M // (1 + int(eta))
        n2 = GAP_M - n1
        S_a = build_decay_cbm_S(n1, n2, S_IN, S_OUT, alpha=a)
        rows_a.append(compute_lemma04_row(S_a, n1, n2, S_IN, S_OUT, a))
    lemma_curves[a] = pd.DataFrame(rows_a).sort_values("eta")

# Semantic palette: α=1.0 is the flat reference (neutral gray); shrinking α
# moves through blue → orange → red as the bound is stressed harder.
ALPHA_COLORS = {1.0: "#4b5563", 0.99: "#1d4ed8", 0.95: "#ea580c", 0.90: "#b91c1c"}

# ──────────────────────────────────────────────────────────────────────
# Figure A — 2x2 heatmaps of S(α) at the demo (η, m). Square layout.
# ──────────────────────────────────────────────────────────────────────
v_lo = float(min(S.min() for S in demo_S.values()))
v_hi = float(max(S.max() for S in demo_S.values()))

fig_h, axes_h = plt.subplots(2, 2, figsize=(7, 7), constrained_layout=True)
im = None
for ax, a in zip(axes_h.flat, ALPHAS_DEMO):
    im = ax.imshow(demo_S[a], cmap="viridis", aspect="equal", vmin=v_lo, vmax=v_hi)
    ax.axhline(n1_demo - 0.5, color="white", lw=0.8)
    ax.axvline(n1_demo - 0.5, color="white", lw=0.8)
    ax.set_title(fr"$\alpha = {a}$", color=ALPHA_COLORS[a], fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
fig_h.colorbar(im, ax=axes_h.ravel().tolist(), fraction=0.04, pad=0.02, shrink=0.9)
# fig_h.suptitle(fr"$S$ at $(\eta={DEMO_ETA},\, m={DEMO_M})$ — $\alpha$ sweep", y=1.02)
plt.show()

# ──────────────────────────────────────────────────────────────────────
# Figure B — Lemma 0.4 slack vs η. Square layout, linear x, no title,
# legend in bottom-right, modular y ticks.
# ──────────────────────────────────────────────────────────────────────
fig_s, ax_s = plt.subplots(figsize=(7, 7), constrained_layout=True)
for a in ALPHAS_DEMO:
    sub = lemma_curves[a]
    ax_s.plot(sub["eta"], sub["gap_slack"], "-o", color=ALPHA_COLORS[a],
              lw=2, markersize=8, label=fr"$\alpha={a}$")
ax_s.set_xlabel(r"$\eta$ (clan-size imbalance)")
ax_s.set_ylabel(
    r"slack  $=\; \Delta\lambda_{\mathrm{emp}} \;-\; "
    r"(n_{\min}\rho \,-\, \eta\, n_{\min}\, S_{\mathrm{out}}^{\max})$"
)
ax_s.set_xticks(ETA_SLACK)
ax_s.yaxis.set_major_locator(MaxNLocator(nbins=8, steps=[1, 2, 5, 10]))
ax_s.legend(fontsize=10, loc="lower right")
ax_s.grid(True, alpha=0.3)
plt.show()

all_slacks = pd.concat(lemma_curves.values())["gap_slack"]
assert (all_slacks >= -1e-9).all(), all_slacks[all_slacks < 0]
print(f"α-sweep gap_slack range: [{all_slacks.min():.4f}, {all_slacks.max():.4f}]")
df_demo[["D_max", "S_in_min", "rho", "gap_emp", "gap_lb", "gap_slack"]]